# Flow and tide boundaries

**What this shows:** the boundary settings for the flow and water level, on each
side of the grid.

**Prerequisites:** [Tutorial 2: Defining the model grid](../tutorial/02_model_grid.ipynb)
(for the names of the grid sides) and
[Tutorial 5: Choosing model settings](../tutorial/05_model_settings.ipynb).

**You will learn:**

- which flow boundary types exist for the offshore, back and lateral boundaries
- how lateral wave boundaries avoid shadow zones
- how water levels are applied with `tideloc`, and how this interacts with tide
  forcing

**Data used:** none.

## Setup

In [1]:
from rompy.logging import config as logging_config
from rompy_xbeach.components.boundary.parameters import (
    FlowBoundaryConditions,
    TideBoundaryConditions,
)

logging_config.update(level="WARNING")  # rompy logs every step, show warnings only


def show(component):
    """Print the XBeach parameters a component writes to params.txt."""
    params = component.get(destdir=None)
    for key, value in params.items():
        print(f"{key} = {value}")
    if not params:
        print("(nothing written, XBeach defaults apply)")

## 1. Flow boundaries

The four sides of the grid are `front` (offshore), `back` (landward), and `left` and
`right` (lateral):

| Side | Options | XBeach default |
|---|---|---|
| `front` | `abs_2d`, `abs_1d`, `wall`, `wlevel`, `nonh_1d`, `waveflume` | `abs_2d` |
| `back` | `abs_2d`, `abs_1d`, `wall`, `wlevel` | `abs_2d` |
| `left`, `right` | `neumann`, `wall`, `no_advec`, `neumann_v`, `abs_1d` | `neumann` |

`abs_2d` and `abs_1d` are absorbing-generating boundaries that let waves leave the
domain, `wall` blocks all flow, `wlevel` imposes a water level, and `neumann` assumes
no alongshore gradient.

In [2]:
show(FlowBoundaryConditions())

(nothing written, XBeach defaults apply)


A beach backed by a dune or seawall, with a closed landward boundary:

In [3]:
show(
    FlowBoundaryConditions(front="abs_2d", back="wall", left="neumann", right="neumann")
)

front = abs_2d
back = wall
left = neumann
right = neumann


For a laboratory flume, the offshore boundary acts as a wave maker and the sides are
walls:

In [4]:
show(
    FlowBoundaryConditions(front="waveflume", back="abs_1d", left="wall", right="wall")
)

front = waveflume
back = abs_1d
left = wall
right = wall


## 2. Lateral wave boundaries

With the default `neumann`, oblique waves can leave a shadow zone near the lateral
boundaries. `wavecrest` assumes no gradient along the wave crests, which reduces this.
`cyclic` connects the two sides for alongshore-uniform tests.

In [5]:
show(FlowBoundaryConditions(lateralwave="wavecrest"))

lateralwave = wavecrest


## 3. Water level boundaries

`tideloc` sets how many water level signals are applied:

| `tideloc` | Water level |
|---|---|
| `0` | Constant level `zs0` everywhere, no tide file |
| `1` | One signal along the offshore boundary |
| `2` | Two signals, offshore and landward (`paulrevere="land"`) or at the two offshore corners (`"sea"`) |
| `4` | One signal at each corner |

XBeach defaults to `tideloc=2`, which needs a tide file. Without tide forcing, set
`tideloc=0`:

In [6]:
show(TideBoundaryConditions(tideloc=0, zs0=0.3))

tideloc = 0
zs0 = 0.3


`tidetype` controls how the water level is imposed: `velocity` (default) through the
flow velocities, `instant` directly on the water level, or `hybrid`.

In [7]:
show(TideBoundaryConditions(tidetype="instant"))

tidetype = instant


### Interaction with water level forcing

The water level forcing classes in `input.tide` write a single timeseries and set
`tideloc=1` themselves (see [water levels](water_levels.ipynb)). The
`tide_boundary` parameters are applied after the forcing, so a `tideloc` set there
overrides the forcing. When you use water level forcing, leave `tideloc` unset and
use `tide_boundary` only for settings such as `tidetype`.

## Summary

- `FlowBoundaryConditions` sets the flow boundary on each side, plus the lateral wave
  boundary.
- `TideBoundaryConditions` sets how water levels are applied. Use `tideloc=0` without
  tide forcing, and leave it unset with tide forcing.
- Wave boundaries are set separately, through `input.wave`.